# InSiSo metal-nut defect detection — Colab training notebook

Runs the full pipeline from the repository's scripts: dataset download → audit → frozen split →
mask-to-box labels → training (baseline + improvement) → held-out evaluation → failure gallery →
ONNX export and parity check → artefact packaging.

* Every step calls the same `scripts/*.py` used locally, so results are reproducible outside Colab.
* Checkpoints are written to Google Drive (if mounted) so an interrupted session can **resume**.
* CPU latency measured here is **Colab CPU**, not the deployment CPU. Run `scripts/benchmark_cpu.py`
  on the target machine for deployment numbers.

## 0. Runtime check

In [ ]:
import subprocess, sys, platform
print(platform.platform(), sys.version)
try:
    import torch
    print("torch", torch.__version__, "CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet")
DEVICE = "0" if "torch" in sys.modules and torch.cuda.is_available() else "cpu"
print("training device ->", DEVICE)

## 1. Get the code and (optionally) persistent storage

In [ ]:
import os
REPO_URL = "https://github.com/<your-user>/insiso-metal-nut-edge-vision.git"  # <-- set this
if not os.path.exists("insiso-metal-nut-edge-vision"):
    !git clone $REPO_URL
%cd insiso-metal-nut-edge-vision

# Persist runs/ and models/ on Google Drive so training can resume after a disconnect.
PERSIST = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/insiso-metal-nut"
    os.makedirs(PERSIST + "/runs", exist_ok=True); os.makedirs(PERSIST + "/models", exist_ok=True)
    for d in ("runs", "models"):
        if os.path.isdir(d) and not os.path.islink(d):
            !cp -rn {d}/. {PERSIST}/{d}/ && rm -rf {d}
        if not os.path.exists(d):
            os.symlink(f"{PERSIST}/{d}", d)
except Exception as e:
    print("Drive not mounted; checkpoints are lost if the session ends:", e)

## 2. Dependencies (Colab already provides a CUDA PyTorch)

In [ ]:
!pip -q install ultralytics==8.4.188 onnx==1.23.2 onnxslim==0.1.98 onnxruntime==1.31.0 psutil py-cpuinfo pytest

## 3. Dataset download, verification and audit
`--allow-mirror` falls back to a pinned unofficial mirror if MVTec's link is down; provenance is recorded either way.

In [ ]:
!python scripts/download_dataset.py --allow-mirror
!cat data/raw/PROVENANCE.json
!python scripts/audit_dataset.py

## 4. Frozen split + mask-to-box labels + QA panels + tests

In [ ]:
!python scripts/prepare_data.py --qa
!python scripts/prepare_data.py --exclude-category scratch --name loco_scratch
!python -m pytest -q tests

## 5. Train baseline (resumes automatically if a checkpoint exists)

In [ ]:
import os
resume = "--resume" if os.path.exists("runs/train/baseline/weights/last.pt") else ""
!python scripts/train.py --config configs/baseline.yaml --device $DEVICE $resume

## 6. Evaluate: threshold chosen on val, then frozen and applied to test

In [ ]:
!python scripts/evaluate.py --model models/baseline.pt --split val
!python scripts/evaluate.py --model models/baseline.pt --split test
!python scripts/analyze_failures.py --name baseline --split test

## 7a. E1: recall-first operating point (no retraining; threshold from val only)

In [ ]:
!python scripts/evaluate.py --model models/baseline.pt --name baseline_zfr --threshold-rule zero_fpr --split val
!python scripts/evaluate.py --model models/baseline.pt --name baseline_zfr --split test
!python scripts/analyze_failures.py --name baseline_zfr --split test

## 7b. E2: merged-fragment annotation policy (one factor changed — see configs/improvement.yaml)
Scored against the SAME per-component reference labels as the baseline, and additionally against merged labels.

In [ ]:
!python scripts/prepare_data.py --policy merged --merge-gap 20 --name merged20
resume = "--resume" if os.path.exists("runs/train/improvement/weights/last.pt") else ""
!python scripts/train.py --config configs/improvement.yaml --device $DEVICE $resume
for name, rule in (("improvement", "max_f1"), ("improvement_zfr", "zero_fpr")):
    !python scripts/evaluate.py --model models/improvement.pt --name {name} --threshold-rule {rule} --gt-dataset baseline --split val
    !python scripts/evaluate.py --model models/improvement.pt --name {name} --gt-dataset baseline --split test
!python scripts/evaluate.py --model models/baseline.pt --name baseline_vs_mergedGT --threshold-from baseline_zfr --gt-dataset merged20 --split test
!python scripts/evaluate.py --model models/improvement.pt --name improvement_vs_mergedGT --threshold-from improvement_zfr --gt-dataset merged20 --split test
!python scripts/analyze_failures.py --name improvement_zfr --split test

## 7c. Leave-one-defect-category-out (scratch) and normal-only PatchCore

In [ ]:
resume = "--resume" if os.path.exists("runs/train/loco_scratch/weights/last.pt") else ""
!python scripts/train.py --config configs/loco_scratch.yaml --device $DEVICE $resume
!python scripts/evaluate.py --model models/loco_scratch.pt --dataset loco_scratch --gt-dataset loco_scratch --name loco_scratch --threshold-rule zero_fpr --split val
!python scripts/evaluate.py --model models/loco_scratch.pt --dataset loco_scratch --gt-dataset loco_scratch --name loco_scratch --split test
!python scripts/evaluate.py --model models/loco_scratch.pt --dataset loco_scratch --gt-dataset loco_scratch --name loco_scratch --split unseen
!python scripts/anomaly_baseline.py
!python tools/patchcore_recall_first.py

## 8. Seed repeats (variance check; not run in the submitted results)

In [ ]:
!python scripts/run_seeds.py --configs configs/baseline.yaml configs/improvement.yaml --seeds 1 2 --device $DEVICE

## 9. ONNX export + parity test + Colab-CPU latency (NOT the deployment CPU)

In [ ]:
!python scripts/export_onnx.py --weights models/baseline.pt
!python scripts/verify_onnx.py --weights models/baseline.pt --onnx models/baseline.onnx --conf 0.11
!python scripts/deployment_variants.py --weights models/baseline.pt --sizes 640 480 320 --int8
!python scripts/benchmark_cpu.py --onnx models/baseline.onnx --label colab-cpu --iters 100

## 10. Visualise a few results

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("reports/failures/baseline_test/*.jpg"))[:5]:
    display(Image(p, width=900))

## 11. Package artefacts

In [ ]:
!zip -qr insiso_artifacts.zip models/*.onnx models/*.onnx.json artifacts reports manifests configs
if PERSIST:
    !cp insiso_artifacts.zip {PERSIST}/
print("done")